# T01 — Data Loading
**Goal:** load all 4 FD subsets, combine into unified DataFrames, save as `train_loaded.csv` / `test_loaded.csv`  
**Output:** `data/loaded/train_loaded.csv`, `data/loaded/test_loaded.csv`  
**Next:** T02_rul_computation.ipynb

In [ ]:
import sys
import os
from pathlib import Path

# add project root to sys.path so src/ imports work from any working directory
ROOT = Path(os.getcwd()).resolve().parents[1]  # experiments/01_data_pipeline/ -> project root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")

In [ ]:
import pandas as pd
from src.preprocessing.load_data import load_all_train, load_all_test

DATA_DIR = ROOT / "data"
RAW_DIR  = DATA_DIR / "raw"
LOADED_DIR = DATA_DIR / 'loaded'
LOADED_DIR.mkdir(parents=True, exist_ok=True)

## Load training data

In [ ]:
print("Loading training files...")
train = load_all_train(RAW_DIR)
train.head(3)

## Load test data + RUL ground truth

In [ ]:
print("Loading test files...")
test = load_all_test(RAW_DIR)
test.head(3)

## Verify structure

In [ ]:
# expected totals from CMAPSS documentation
EXPECTED_TRAIN = {4:249}  # your version
EXPECTED_TEST  = {4:248}

assert train["engine_id"].nunique() == sum(EXPECTED_TRAIN.values())
assert test["engine_id"].nunique()  == sum(EXPECTED_TEST.values())
# engine IDs must be non-overlapping between train and test
# (they're in separate DataFrames so offsets are independent — this is fine)

# rul_last must exist in test at this stage (needed by T02)
assert "rul_last" in test.columns, "test DataFrame missing 'rul_last' column"

# no NaN values in either set
assert train.isnull().sum().sum() == 0, "NaN values in train"
assert test.isnull().sum().sum() == 0, "NaN values in test"

print(f"[PASS] train: {train.shape}, {train['engine_id'].nunique()} engines")
print(f"[PASS] test:  {test.shape},  {test['engine_id'].nunique()} engines")
print(f"[PASS] no NaN values")

## Per-subset engine counts

In [ ]:
print("Train engines per subset:")
print(train.groupby("dataset_id")["engine_id"].nunique().to_string())
print("\nTest engines per subset:")
print(test.groupby("dataset_id")["engine_id"].nunique().to_string())



## Save

In [ ]:
train.drop(columns='engine_id', axis=1)
test.drop(columns='engine_id', axis=1)

In [ ]:
train.to_csv(LOADED_DIR / "train_loaded.csv", index=False)
test.to_csv(LOADED_DIR / "test_loaded.csv", index=False)
print(f"Saved: {LOADED_DIR / 'train_loaded.csv'}")
print(f"Saved: {LOADED_DIR / 'test_loaded.csv'}")